# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Content can lose search performance as it becomes stale

The FlyRank material describes stale content as an important refresh signal. The operational logic uses measurable signals such as page age, impressions and recent performance to identify pages for review.

**Methodology question:** The label used in my model is `is_declining_label`, which is derived from `trend_direction == "down"`. I should therefore treat the result as evidence of measured decline in this dataset, not proof that staleness itself caused the decline.

### Finding 2 — Search performance depends on multiple measurable signals

The FlyRank Refresh framework uses several signals together, including impressions, clicks, CTR, position, engagement and content age, rather than relying on one metric alone.

**Methodology question:** My model is useful for ranking pages for review, but the validation design must keep clients separated between training and testing. Otherwise, similar pages from the same client could make the model appear stronger than it really is.

These questions do not reject the findings. They identify what the available validation design can and cannot support.

In [2]:
# =========================================================
# ML-09 — SETUP + BASIC LABEL CHECK
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd



# Required fields for this audit
required = [
    "client_id",
    "content_id",
    "is_declining_label",
    "trend_direction",
]

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise KeyError(
        f"Missing required columns: {missing}"
    )

print("\nLabel definition check:")

label_check = pd.crosstab(
    df["trend_direction"],
    df["is_declining_label"],
    dropna=False
)

display(label_check)

# Confirm the documented label relationship
expected_label = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

label_matches = (
    expected_label == df["is_declining_label"].astype(int)
).mean()

print(
    f"Label matches trend_direction == 'down': "
    f"{label_matches:.3f}"
)

assert label_matches == 1.0, (
    "The label definition does not match "
    "trend_direction == 'down'."
)


Label definition check:


is_declining_label,0,1
trend_direction,,
down,0,16262
flat,1152,0
new,2236,0
stable,5962,0
up,4388,0


Label matches trend_direction == 'down': 1.000


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*



My Week-5 model already used a grouped client-holdout split. For this validation audit I repeat the experiment with a fresh grouped split using a different random seed.

The important comparison is not whether the exact Precision@50 number stays identical. The question is whether the model still provides useful ranking performance when completely unseen clients are used for evaluation.

A meaningful drop would be a warning that the Week-5 result was sensitive to the particular split.

In [3]:
# =========================================================
# 2. HONEST VALIDATION — FRESH CLIENT HOLDOUT
# =========================================================

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

TARGET = "is_declining_label"
GROUP = "client_id"

validation_df = df.dropna(
    subset=[TARGET, GROUP]
).copy()

validation_df[TARGET] = (
    validation_df[TARGET]
    .astype(int)
)

# ---------------------------------------------------------
# IMPORTANT: remove target, label-derived fields and IDs
# ---------------------------------------------------------

DROP_COLUMNS = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id",
}

feature_cols = [
    c for c in validation_df.columns
    if c not in DROP_COLUMNS
]

X = validation_df[feature_cols]
y = validation_df[TARGET]
groups = validation_df[GROUP]

# ---------------------------------------------------------
# Fresh grouped split
# ---------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=2026
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(
    validation_df.iloc[train_idx][GROUP]
)

test_clients = set(
    validation_df.iloc[test_idx][GROUP]
)

overlap = train_clients.intersection(
    test_clients
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print(
    "Training clients:",
    len(train_clients)
)

print(
    "Test clients:",
    len(test_clients)
)

print(
    "Client overlap:",
    len(overlap)
)

assert len(overlap) == 0, (
    "Validation leakage: clients overlap."
)

# ---------------------------------------------------------
# Preprocessing
# ---------------------------------------------------------

numeric_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        )
    )
])

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_pipeline,
        numeric_features
    ),
    (
        "cat",
        categorical_pipeline,
        categorical_features
    ),
])

# ---------------------------------------------------------
# Same model family as ML-08
# ---------------------------------------------------------

rf_validation = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

validation_model = Pipeline([
    (
        "preprocess",
        preprocessor
    ),
    (
        "model",
        rf_validation
    )
])

print("\nTraining validation model...")

validation_model.fit(
    X_train,
    y_train
)

# ---------------------------------------------------------
# Score test set
# ---------------------------------------------------------

validation_scores = (
    validation_model
    .predict_proba(X_test)[:, 1]
)

validation_results = validation_df.iloc[
    test_idx
][
    [
        "content_id",
        "client_id",
        TARGET
    ]
].copy()

validation_results[
    "model_score"
] = validation_scores

validation_results = (
    validation_results
    .sort_values(
        "model_score",
        ascending=False
    )
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Precision@50
# ---------------------------------------------------------

k = min(50, len(validation_results))

precision_at_50_after = (
    validation_results
    .head(k)[TARGET]
    .mean()
)

print(
    "\nFresh client-holdout Precision@50:",
    round(precision_at_50_after, 4)
)

print(
    "Positive pages in top 50:",
    int(
        validation_results
        .head(k)[TARGET]
        .sum()
    ),
    "/",
    k
)

# ---------------------------------------------------------
# Base rate
# ---------------------------------------------------------

base_rate = y_test.mean()

print(
    "Test-set declining-page base rate:",
    round(base_rate, 4)
)

# ---------------------------------------------------------
# Compare with ML-08 result if variable exists
# ---------------------------------------------------------

print("\nBEFORE / AFTER")

if "precision_at_50" in globals():

    precision_at_50_before = precision_at_50

    comparison = pd.DataFrame({
        "evaluation": [
            "ML-08 original split",
            "ML-09 fresh client split"
        ],
        "precision_at_50": [
            precision_at_50_before,
            precision_at_50_after
        ]
    })

    comparison["difference"] = (
        comparison["precision_at_50"]
        - precision_at_50_before
    )

    display(comparison)

else:

    print(
        "ML-08 precision_at_50 variable was not "
        "available in this notebook."
    )

    print(
        "Current ML-09 Precision@50:",
        round(precision_at_50_after, 4)
    )

Training rows: 26119
Test rows: 3881
Training clients: 25
Test clients: 7
Client overlap: 0

Training validation model...

Fresh client-holdout Precision@50: 1.0
Positive pages in top 50: 50 / 50
Test-set declining-page base rate: 0.6094

BEFORE / AFTER
ML-08 precision_at_50 variable was not available in this notebook.
Current ML-09 Precision@50: 1.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I checked the final feature set for fields that directly define the target or identify the evaluation group.

The target is derived from `trend_direction`, so `trend_direction` and `trend_pct` are excluded.

`content_id` and `client_id` are identifiers rather than predictive signals and are excluded from the model.

The audit below also checks for suspicious feature names containing terms associated with labels, outcomes, trends, or future information.

In [4]:
# =========================================================
# 3. LEAKAGE AUDIT
# =========================================================

# The final feature set used by the validation model
final_features = feature_cols

print("Number of final model features:", len(final_features))

print("\nFinal model features:")
for i, col in enumerate(final_features, start=1):
    print(f"{i:02d}. {col}")

# ---------------------------------------------------------
# Directly forbidden columns
# ---------------------------------------------------------

forbidden = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id",
}

leaked_direct = sorted(
    set(final_features).intersection(
        forbidden
    )
)

print("\nDirect leakage / identifier columns found:")
print(leaked_direct)

assert leaked_direct == [], (
    "Forbidden columns found in final feature set."
)

# ---------------------------------------------------------
# Name-based warning scan
# ---------------------------------------------------------

warning_terms = [
    "label",
    "target",
    "outcome",
    "trend_direction",
    "trend_pct",
    "future",
    "next_period",
    "next_30",
    "next_90",
    "post",
    "after",
]

suspicious = []

for col in final_features:
    col_lower = col.lower()

    if any(
        term in col_lower
        for term in warning_terms
    ):
        suspicious.append(col)

print("\nPotentially suspicious feature names:")
print(suspicious)

# These require human inspection rather than automatic
# rejection because some feature names can be legitimate.

Number of final model features: 47

Final model features:
01. search_volume
02. competition
03. competition_level
04. cpc
05. content_type
06. main_intent
07. word_count
08. char_count
09. provider_used
10. model_used
11. impressions_90d
12. clicks_90d
13. pageviews_90d
14. sessions_90d
15. users_90d
16. engaged_sessions_90d
17. ai_sessions_90d
18. scroll_events_90d
19. days_with_impressions
20. days_with_sessions
21. impressions_last_30d
22. clicks_last_30d
23. sessions_last_30d
24. impressions_prev_30d
25. clicks_prev_30d
26. sessions_prev_30d
27. content_age_days
28. age_tier
29. age_tier_order
30. days_since_last_update
31. freshness_tier
32. word_count_tier
33. char_count_tier
34. ctr
35. avg_position
36. engagement_rate
37. scroll_rate
38. ai_traffic_pct
39. impression_tier
40. position_tier
41. log_impressions_90d
42. log_clicks_90d
43. log_sessions_90d
44. log_ai_sessions_90d
45. has_clicks
46. has_ai_sessions
47. measurable_opportunity

Direct leakage / identifier columns foun

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original bold claim

"The Random Forest predicts which pages will decline and identifies the pages that Google needs to fix."

### Safer claim

"On the anonymized dataset, the Random Forest produced higher Precision@50 than the hand-written baseline under a grouped client-holdout evaluation. This is an observed, measured result for decline detection in this dataset. The model should be treated as decision-support for prioritizing pages for review; it does not establish causality or predict Google's ranking algorithm."

In [5]:
# =========================================================
# 4. CLAIM SUPPORT CHECK
# =========================================================

print("CLAIM AUDIT")
print("=" * 60)

print(
    "Observed metric: Precision@50"
)

print(
    "Validation design: grouped client holdout"
)

print(
    "Client overlap:",
    len(overlap)
)

print(
    "ML-09 Precision@50:",
    round(
        precision_at_50_after,
        4
    )
)

print(
    "\nSafe interpretation:"
)

print(
    "The measured result supports using the model "
    "as decision-support for prioritizing pages "
    "for review in this dataset."
)

print(
    "\nNot supported:"
)

print(
    "The validation does not prove causality, "
    "does not prove that the model generalizes "
    "to every website, and does not predict "
    "Google's ranking algorithm."
)

assert len(overlap) == 0

CLAIM AUDIT
Observed metric: Precision@50
Validation design: grouped client holdout
Client overlap: 0
ML-09 Precision@50: 1.0

Safe interpretation:
The measured result supports using the model as decision-support for prioritizing pages for review in this dataset.

Not supported:
The validation does not prove causality, does not prove that the model generalizes to every website, and does not predict Google's ranking algorithm.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.